# Transformers: Architecture, Attention Mechanisms, and Applications

## 1. Import Required Libraries



In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import math
import pandas as pd
from torch.nn import TransformerEncoder, TransformerEncoderLayer
from torch.utils.data import DataLoader, Dataset

# Set random seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Check if GPU is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")



## 2. Understanding Transformer Architecture

### What are Transformers?

Transformers are a type of neural network architecture introduced in the paper "Attention Is All You Need" (Vaswani et al., 2017). They revolutionized Natural Language Processing and have since been applied to various domains including computer vision, time series analysis, and more.

### Key Innovations of Transformers:

1. **Self-Attention Mechanism**: Allows the model to weigh the importance of different words in a sequence relative to each other
2. **Parallelization**: Unlike RNNs, transformers process all elements of a sequence in parallel
3. **Long-range Dependencies**: Effectively capture relationships between distant elements in a sequence
4. **Positional Encoding**: Maintain sequence order information without recurrence

### Why Transformers Over RNNs/LSTMs?

1. **Parallelization**: Transformers can be trained much faster than RNNs as they don't require sequential processing
2. **No vanishing/exploding gradients**: The architecture avoids issues common in RNNs
3. **Direct connections**: Every position can attend to every other position with a direct path
4. **Better performance**: Transformers generally achieve state-of-the-art results in many tasks



In [ ]:
# Let's visualize the high-level transformer architecture
def plot_transformer_architecture():
    fig, ax = plt.subplots(figsize=(12, 10))
    
    # Define components positions
    components = {
        "Input Embedding": (0.5, 0.1),
        "Positional Encoding": (0.5, 0.2),
        "Encoder": (0.3, 0.5),
        "Decoder": (0.7, 0.5),
        "Output Probabilities": (0.5, 0.9),
    }
    
    # Define sub-components
    encoder_components = {
        "Multi-Head\nAttention": (0.3, 0.4),
        "Add & Norm": (0.3, 0.5),
        "Feed Forward": (0.3, 0.6),
        "Add & Norm 2": (0.3, 0.7),
    }
    
    decoder_components = {
        "Masked\nMulti-Head\nAttention": (0.7, 0.3),
        "Add & Norm": (0.7, 0.4),
        "Multi-Head\nAttention": (0.7, 0.5),
        "Add & Norm 2": (0.7, 0.6),
        "Feed Forward": (0.7, 0.7),
        "Add & Norm 3": (0.7, 0.8),
    }
    
    # Draw main components as boxes
    for name, (x, y) in components.items():
        if name == "Encoder" or name == "Decoder":
            rect = plt.Rectangle((x-0.2, y-0.3), 0.4, 0.6, fill=True, 
                                color='lightgray' if name == "Encoder" else 'lightblue', 
                                alpha=0.3)
            ax.add_patch(rect)
            ax.text(x, y, name, ha='center', va='center', fontsize=14, fontweight='bold')
        else:
            rect = plt.Rectangle((x-0.15, y-0.04), 0.3, 0.08, fill=True, 
                                color='lightgreen', alpha=0.8)
            ax.add_patch(rect)
            ax.text(x, y, name, ha='center', va='center', fontsize=12)
    
    # Draw encoder sub-components
    for name, (x, y) in encoder_components.items():
        rect = plt.Rectangle((x-0.12, y-0.03), 0.24, 0.06, fill=True, 
                            color='orange', alpha=0.6)
        ax.add_patch(rect)
        ax.text(x, y, name, ha='center', va='center', fontsize=10)
    
    # Draw decoder sub-components
    for name, (x, y) in decoder_components.items():
        rect = plt.Rectangle((x-0.12, y-0.03), 0.24, 0.06, fill=True, 
                            color='orange', alpha=0.6)
        ax.add_patch(rect)
        ax.text(x, y, name, ha='center', va='center', fontsize=10)
    
    # Connect components with arrows
    ax.arrow(0.5, 0.1, 0, 0.06, head_width=0.02, head_length=0.02, fc='k', ec='k')
    ax.arrow(0.5, 0.2, -0.1, 0.1, head_width=0.02, head_length=0.02, fc='k', ec='k')
    ax.arrow(0.5, 0.2, 0.1, 0.05, head_width=0.02, head_length=0.02, fc='k', ec='k')
    ax.arrow(0.3, 0.7, 0, 0.1, head_width=0.02, head_length=0.02, fc='k', ec='k')
    ax.arrow(0.3, 0.8, 0.15, 0.05, head_width=0.02, head_length=0.02, fc='k', ec='k')
    ax.arrow(0.7, 0.8, -0.1, 0.06, head_width=0.02, head_length=0.02, fc='k', ec='k')
    
    # Cross-connections
    ax.arrow(0.4, 0.5, 0.15, 0, head_width=0.02, head_length=0.02, fc='k', ec='k')
    
    # Add annotation for Nx
    ax.text(0.3, 0.25, "Nx", ha='center', va='center', fontsize=14, 
            bbox=dict(boxstyle="circle", fc="white", ec="k"))
    ax.text(0.7, 0.25, "Nx", ha='center', va='center', fontsize=14,
            bbox=dict(boxstyle="circle", fc="white", ec="k"))
    
    # Set plot limits and remove axes
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis('off')
    
    plt.title("Transformer Architecture", fontsize=16)
    plt.tight_layout()
    plt.show()

plot_transformer_architecture()



## 3. The Attention Mechanism

### What is Attention?

Attention is a mechanism that allows a model to focus on specific parts of the input sequence when producing an output. It was initially developed to address the limitations of encoder-decoder architectures in handling long sequences.

### Self-Attention

Self-attention, also known as intra-attention, relates different positions of a single sequence to compute a representation of the sequence. It allows each position to attend to all positions within the sequence.

### The Mathematics of Attention

The scaled dot-product attention is computed as:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

Where:
- Q (Query): What we're looking for
- K (Key): What we match against
- V (Value): What we retrieve if there's a match
- $d_k$: Dimension of keys (scaling factor to prevent softmax from having extremely small gradients)



In [ ]:
def visualize_attention_mechanism():
    # Setup the figure
    fig, ax = plt.subplots(figsize=(10, 7))
    
    # Define the positions
    query_pos = (0.2, 0.8)
    key_pos = (0.2, 0.5)
    value_pos = (0.2, 0.2)
    
    matmul1_pos = (0.5, 0.65)
    scale_pos = (0.5, 0.55)
    softmax_pos = (0.5, 0.45)
    matmul2_pos = (0.5, 0.35)
    
    output_pos = (0.8, 0.5)
    
    # Draw boxes for Q, K, V
    for name, pos, color in [('Query (Q)', query_pos, 'lightblue'), 
                             ('Key (K)', key_pos, 'lightgreen'), 
                             ('Value (V)', value_pos, 'lightsalmon')]:
        rect = plt.Rectangle((pos[0]-0.1, pos[1]-0.07), 0.2, 0.14, fill=True, 
                            color=color, alpha=0.7)
        ax.add_patch(rect)
        ax.text(pos[0], pos[1], name, ha='center', va='center', fontsize=12)
    
    # Draw operations
    for name, pos in [('MatMul', matmul1_pos),
                      ('Scale', scale_pos),
                      ('Softmax', softmax_pos),
                      ('MatMul', matmul2_pos)]:
        circle = plt.Circle(pos, 0.07, fill=True, color='lightgray', alpha=0.7)
        ax.add_patch(circle)
        ax.text(pos[0], pos[1], name, ha='center', va='center', fontsize=10)
    
    # Draw output
    rect = plt.Rectangle((output_pos[0]-0.1, output_pos[1]-0.07), 0.2, 0.14, fill=True, 
                        color='yellow', alpha=0.7)
    ax.add_patch(rect)
    ax.text(output_pos[0], output_pos[1], 'Output', ha='center', va='center', fontsize=12)
    
    # Connect with arrows
    ax.arrow(0.3, 0.8, 0.13, -0.1, head_width=0.02, head_length=0.02, fc='k', ec='k')
    ax.arrow(0.3, 0.5, 0.13, 0.08, head_width=0.02, head_length=0.02, fc='k', ec='k')
    ax.arrow(0.5, 0.58, 0, -0.06, head_width=0.02, head_length=0.02, fc='k', ec='k')
    ax.arrow(0.5, 0.48, 0, -0.06, head_width=0.02, head_length=0.02, fc='k', ec='k')
    ax.arrow(0.3, 0.2, 0.13, 0.08, head_width=0.02, head_length=0.02, fc='k', ec='k')
    ax.arrow(0.57, 0.35, 0.13, 0.08, head_width=0.02, head_length=0.02, fc='k', ec='k')
    
    # Add formula
    ax.text(0.5, 0.1, r"$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$", 
            ha='center', va='center', fontsize=14)
    
    # Set plot limits and remove axes
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis('off')
    
    plt.title("Scaled Dot-Product Attention", fontsize=16)
    plt.tight_layout()
    plt.show()

visualize_attention_mechanism()



## 4. Multi-Head Attention

Instead of performing a single attention function, multi-head attention performs the attention function in parallel on different projected versions of queries, keys, and values.

This allows the model to:
- Attend to information from different representation subspaces
- Capture different types of relationships within the data
- Learn patterns at different positions

The multi-head attention is computed as:

$$\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, \ldots, \text{head}_h)W^O$$

Where each head is:

$$\text{head}_i = \text{Attention}(QW_i^Q, KW_i^K, VW_i^V)$$



In [ ]:
def visualize_multihead_attention():
    # Setup the figure
    fig, ax = plt.subplots(figsize=(12, 8))
    
    # Define the positions
    input_pos = (0.5, 0.1)
    
    head_positions = [(0.2, 0.4), (0.4, 0.4), (0.6, 0.4), (0.8, 0.4)]
    
    concat_pos = (0.5, 0.6)
    linear_pos = (0.5, 0.8)
    output_pos = (0.5, 0.9)
    
    # Draw input
    rect = plt.Rectangle((input_pos[0]-0.2, input_pos[1]-0.05), 0.4, 0.1, fill=True, 
                        color='lightgreen', alpha=0.7)
    ax.add_patch(rect)
    ax.text(input_pos[0], input_pos[1], 'Input', ha='center', va='center', fontsize=12)
    
    # Draw attention heads
    for i, pos in enumerate(head_positions):
        circle = plt.Circle(pos, 0.08, fill=True, color='lightblue', alpha=0.7)
        ax.add_patch(circle)
        ax.text(pos[0], pos[1], f'Head {i+1}', ha='center', va='center', fontsize=10)
        
        # Connect input to heads
        ax.arrow(input_pos[0], input_pos[1]+0.05, pos[0]-input_pos[0], pos[1]-input_pos[1]-0.13, 
                head_width=0.02, head_length=0.02, fc='k', ec='k')
    
    # Draw concatenation
    rect = plt.Rectangle((concat_pos[0]-0.2, concat_pos[1]-0.05), 0.4, 0.1, fill=True, 
                        color='lightsalmon', alpha=0.7)
    ax.add_patch(rect)
    ax.text(concat_pos[0], concat_pos[1], 'Concatenate', ha='center', va='center', fontsize=12)
    
    # Connect heads to concatenation
    for pos in head_positions:
        ax.arrow(pos[0], pos[1]+0.08, concat_pos[0]-pos[0], concat_pos[1]-0.05-pos[1]-0.08, 
                head_width=0.02, head_length=0.02, fc='k', ec='k')
    
    # Draw linear projection
    rect = plt.Rectangle((linear_pos[0]-0.15, linear_pos[1]-0.05), 0.3, 0.1, fill=True, 
                        color='lightgray', alpha=0.7)
    ax.add_patch(rect)
    ax.text(linear_pos[0], linear_pos[1], 'Linear', ha='center', va='center', fontsize=12)
    
    # Connect concatenation to linear
    ax.arrow(concat_pos[0], concat_pos[1]+0.05, 0, linear_pos[1]-0.05-concat_pos[1]-0.05, 
            head_width=0.02, head_length=0.02, fc='k', ec='k')
    
    # Draw output
    rect = plt.Rectangle((output_pos[0]-0.15, output_pos[1]-0.05), 0.3, 0.1, fill=True, 
                        color='yellow', alpha=0.7)
    ax.add_patch(rect)
    ax.text(output_pos[0], output_pos[1], 'Output', ha='center', va='center', fontsize=12)
    
    # Connect linear to output
    ax.arrow(linear_pos[0], linear_pos[1]+0.05, 0, output_pos[1]-0.05-linear_pos[1]-0.05, 
            head_width=0.02, head_length=0.02, fc='k', ec='k')
    
    # Set plot limits and remove axes
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis('off')
    
    plt.title("Multi-Head Attention", fontsize=16)
    plt.tight_layout()
    plt.show()

visualize_multihead_attention()



## 5. Positional Encoding

Since transformers don't have any recurrence or convolution, they don't inherently capture the order of the sequence. To incorporate positional information:

1. Positional encodings are added to the input embeddings
2. The encoding uses sine and cosine functions of different frequencies:

$$PE_{(pos, 2i)} = \sin(pos/10000^{2i/d_{model}})$$
$$PE_{(pos, 2i+1)} = \cos(pos/10000^{2i/d_{model}})$$

Where:
- pos: position in the sequence
- i: dimension index
- $d_{model}$: embedding dimension



In [ ]:
def positional_encoding(max_seq_len, d_model):
    """Generate positional encodings for transformer models"""
    pe = torch.zeros(max_seq_len, d_model)
    position = torch.arange(0, max_seq_len, dtype=torch.float).unsqueeze(1)
    div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
    
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    
    return pe

# Visualize positional encoding
def plot_positional_encoding():
    # Generate positional encodings
    max_seq_len = 100
    d_model = 128
    pe = positional_encoding(max_seq_len, d_model)
    
    # Plot as a heatmap
    plt.figure(figsize=(15, 8))
    sns.heatmap(pe.numpy(), cmap='viridis')
    plt.title("Positional Encoding (100 positions × 128 dimensions)")
    plt.xlabel("Encoding Dimension")
    plt.ylabel("Position in Sequence")
    plt.colorbar(label="Value")
    plt.show()
    
    # Plot specific dimensions
    plt.figure(figsize=(15, 6))
    
    # Plot a few dimensions
    for i in [0, 1, 2, 3, 63, 64, 65, 127]:
        plt.plot(pe[:, i].numpy(), label=f'Dimension {i}')
    
    plt.legend()
    plt.title("Positional Encoding Values for Selected Dimensions")
    plt.xlabel("Position in Sequence")
    plt.ylabel("Encoding Value")
    plt.grid(True)
    plt.show()

plot_positional_encoding()



## 6. Encoder-Decoder Architecture

### Encoder

The encoder transforms an input sequence into a continuous representation. Each encoder layer has:

1. **Multi-Head Self-Attention**: Helps the encoder focus on relevant parts of the input
2. **Position-wise Feed-Forward Network**: A fully connected network applied to each position separately

### Decoder

The decoder generates the output sequence. Each decoder layer has:

1. **Masked Multi-Head Self-Attention**: Prevents attending to future positions
2. **Multi-Head Attention**: Attends to the encoder output
3. **Position-wise Feed-Forward Network**: Similar to the encoder

### Layer Normalization and Residual Connections

Both encoder and decoder use:
- **Layer Normalization**: Normalizes the outputs of sub-layers
- **Residual Connections**: Helps with training deeper networks



In [ ]:
# Define a simplified transformer encoder layer
class SimpleTransformerEncoderLayer(nn.Module):
    def __init__(self, d_model, nhead, dim_feedforward=2048, dropout=0.1):
        super(SimpleTransformerEncoderLayer, self).__init__()
        self.self_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout)
        
        # Feed-forward network
        self.linear1 = nn.Linear(d_model, dim_feedforward)
        self.dropout = nn.Dropout(dropout)
        self.linear2 = nn.Linear(dim_feedforward, d_model)
        
        # Normalization layers
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        
        # Dropout
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        
        # Activation
        self.activation = nn.ReLU()
        
    def forward(self, src, src_mask=None, src_key_padding_mask=None):
        # Self-attention block
        src2 = self.self_attn(src, src, src, attn_mask=src_mask, 
                              key_padding_mask=src_key_padding_mask)[0]
        src = src + self.dropout1(src2)
        src = self.norm1(src)
        
        # Feed-forward block
        src2 = self.linear2(self.dropout(self.activation(self.linear1(src))))
        src = src + self.dropout2(src2)
        src = self.norm2(src)
        
        return src

# Define a simplified transformer decoder layer
class SimpleTransformerDecoderLayer(nn.Module):
    def __init__(self, d_model, nhead, dim_feedforward=2048, dropout=0.1):
        super(SimpleTransformerDecoderLayer, self).__init__()
        
        # Self-attention (masked)
        self.self_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout)
        
        # Cross-attention (to encoder outputs)
        self.multihead_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout)
        
        # Feed-forward network
        self.linear1 = nn.Linear(d_model, dim_feedforward)
        self.dropout = nn.Dropout(dropout)
        self.linear2 = nn.Linear(dim_feedforward, d_model)
        
        # Normalization layers
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        
        # Dropout
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.dropout3 = nn.Dropout(dropout)
        
        # Activation
        self.activation = nn.ReLU()
        
    def forward(self, tgt, memory, tgt_mask=None, memory_mask=None,
                tgt_key_padding_mask=None, memory_key_padding_mask=None):
        
        # Self-attention block (with mask to prevent attending to future positions)
        tgt2 = self.self_attn(tgt, tgt, tgt, attn_mask=tgt_mask,
                              key_padding_mask=tgt_key_padding_mask)[0]
        tgt = tgt + self.dropout1(tgt2)
        tgt = self.norm1(tgt)
        
        # Cross-attention block (attending to encoder outputs)
        tgt2 = self.multihead_attn(tgt, memory, memory, attn_mask=memory_mask,
                                  key_padding_mask=memory_key_padding_mask)[0]
        tgt = tgt + self.dropout2(tgt2)
        tgt = self.norm2(tgt)
        
        # Feed-forward block
        tgt2 = self.linear2(self.dropout(self.activation(self.linear1(tgt))))
        tgt = tgt + self.dropout3(tgt2)
        tgt = self.norm3(tgt)
        
        return tgt

print("Transformer Encoder Layer:", SimpleTransformerEncoderLayer(512, 8))
print("\nTransformer Decoder Layer:", SimpleTransformerDecoderLayer(512, 8))



## 7. A Complete Transformer Model

Now let's build a complete transformer model using PyTorch's built-in modules.



In [ ]:
class TransformerModel(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model=512, nhead=8, 
                 num_encoder_layers=6, num_decoder_layers=6, dim_feedforward=2048, 
                 dropout=0.1, max_seq_length=5000):
        super(TransformerModel, self).__init__()
        
        # Embedding layers
        self.src_embedding = nn.Embedding(src_vocab_size, d_model)
        self.tgt_embedding = nn.Embedding(tgt_vocab_size, d_model)
        
        # Positional encoding
        self.pos_encoder = PositionalEncoding(d_model, max_seq_length, dropout)
        
        # Transformer layers
        encoder_layer = nn.TransformerEncoderLayer(d_model, nhead, dim_feedforward, dropout)
        self.encoder = nn.TransformerEncoder(encoder_layer, num_encoder_layers)
        
        decoder_layer = nn.TransformerDecoderLayer(d_model, nhead, dim_feedforward, dropout)
        self.decoder = nn.TransformerDecoder(decoder_layer, num_decoder_layers)
        
        self.out_projection = nn.Linear(d_model, tgt_vocab_size)
        
        self.d_model = d_model
    
    def forward(self, src, tgt, src_mask=None, tgt_mask=None, 
                src_padding_mask=None, tgt_padding_mask=None, memory_mask=None):
        
        # Embed source tokens and add positional encoding
        src = self.src_embedding(src) * math.sqrt(self.d_model)
        src = self.pos_encoder(src)
        
        # Embed target tokens and add positional encoding
        tgt = self.tgt_embedding(tgt) * math.sqrt(self.d_model)
        tgt = self.pos_encoder(tgt)
        
        # Transformer encoder
        memory = self.encoder(src, mask=src_mask, src_key_padding_mask=src_padding_mask)
        
        # Transformer decoder
        output = self.decoder(tgt, memory, tgt_mask=tgt_mask, memory_mask=memory_mask,
                             tgt_key_padding_mask=tgt_padding_mask)
        
        # Project to vocabulary
        output = self.out_projection(output)
        
        return output

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_seq_length=5000, dropout=0.1):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)
        
        # Create positional encoding
        pe = torch.zeros(max_seq_length, d_model)
        position = torch.arange(0, max_seq_length, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0).transpose(0, 1)
        
        # Register buffer (not a parameter, but part of the module)
        self.register_buffer('pe', pe)
        
    def forward(self, x):
        # Add positional encoding
        x = x + self.pe[:x.size(0), :]
        return self.dropout(x)

# Initialize a small toy model
toy_transformer = TransformerModel(
    src_vocab_size=1000,
    tgt_vocab_size=1000,
    d_model=256,
    nhead=8,
    num_encoder_layers=3,
    num_decoder_layers=3,
    dim_feedforward=512,
    dropout=0.1
)

print(toy_transformer)



## 8. Applications of Transformers

Transformers have revolutionized multiple domains:

### Natural Language Processing
- **BERT (Bidirectional Encoder Representations from Transformers)**: Pre-trained language model for tasks like question-answering, sentiment analysis
- **GPT (Generative Pre-trained Transformer)**: Language generation model for text completion, summarization, translation
- **T5 (Text-to-Text Transfer Transformer)**: Unified approach to NLP tasks

### Computer Vision
- **Vision Transformer (ViT)**: Image classification using transformers
- **DETR (Detection Transformer)**: Object detection with transformers
- **Swin Transformer**: Hierarchical vision transformer with shifted windows

### Multi-modal Tasks
- **CLIP**: Connecting text and images
- **DALL-E**: Generating images from text descriptions

### Time Series Analysis
- **Transformers for Forecasting**: Predicting future values in time series
- **Anomaly Detection**: Finding unusual patterns in sequential data



In [ ]:
def plot_transformer_timeline():
    # Define transformer models and their release dates
    models = {
        'Transformer': 2017,
        'BERT': 2018,
        'GPT-2': 2019,
        'T5': 2019,
        'Vision Transformer': 2020,
        'GPT-3': 2020,
        'CLIP': 2021,
        'DALL-E': 2021,
        'GPT-4': 2023,
        'LLaMa': 2023,
        'Claude': 2023,
        'Gemini': 2023
    }
    
    # Create the figure
    fig, ax = plt.subplots(figsize=(12, 6))
    
    # Plot timeline
    years = list(set(models.values()))
    years.sort()
    
    for i, (model, year) in enumerate(models.items()):
        y_pos = i % 2 * 0.4 + 0.3
        color = plt.cm.tab10(i % 10)
        
        # Add model name
        ax.scatter([year], [y_pos], s=100, color=color, zorder=2)
        ax.text(year, y_pos+0.05, model, ha='center', fontsize=10, fontweight='bold')
    
    # Enhance the timeline
    ax.axhline(y=0.5, color='gray', linestyle='-', alpha=0.3, zorder=1)
    
    for year in range(min(years), max(years)+1):
        ax.axvline(x=year, color='gray', linestyle='--', alpha=0.2, zorder=1)
        ax.text(year, 0.1, str(year), ha='center')
    
    # Set axis limits
    ax.set_xlim(min(years)-0.5, max(years)+0.5)
    ax.set_ylim(0, 1)
    
    # Remove y-axis
    ax.yaxis.set_visible(False)
    ax.spines['left'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['top'].set_visible(False)
    
    plt.title("Timeline of Transformer Model Development", fontsize=16)
    plt.tight_layout()
    plt.show()

plot_transformer_timeline()



## 9. Implementing a Simple Transformer for Sequence Prediction

Let's implement a transformer model for time series prediction:



In [ ]:
class TimeSeriesTransformer(nn.Module):
    def __init__(self, input_dim=1, output_dim=1, d_model=64, nhead=4, 
                 num_layers=2, dim_feedforward=256, dropout=0.1):
        super(TimeSeriesTransformer, self).__init__()
        
        # Input projection
        self.input_projection = nn.Linear(input_dim, d_model)
        
        # Positional encoding
        self.pos_encoder = PositionalEncoding(d_model, dropout=dropout)
        
        # Transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, 
            dim_feedforward=dim_feedforward, dropout=dropout
        )
        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer, num_layers=num_layers
        )
        
        # Output projection
        self.output_projection = nn.Linear(d_model, output_dim)
        
        self.d_model = d_model
        
    def forward(self, src, src_mask=None):
        # src shape: [seq_len, batch_size, input_dim]
        
        # Project input to d_model dimensions
        src = self.input_projection(src)
        
        # Scale by sqrt(d_model)
        src = src * math.sqrt(self.d_model)
        
        # Add positional encoding
        src = self.pos_encoder(src)
        
        # Pass through transformer encoder
        output = self.transformer_encoder(src, src_mask)
        
        # Project to output dimension
        output = self.output_projection(output)
        
        # We only need the last time step prediction
        return output[-1]

# Define a function to generate a sequence-to-one mask (optional)
def generate_square_subsequent_mask(sz):
    """Generate a square mask for the sequence."""
    mask = (torch.triu(torch.ones(sz, sz)) == 1).transpose(0, 1)
    mask = mask.float().masked_fill(mask == 0, float('-inf')).masked_fill(mask == 1, float(0.0))
    return mask

# Create synthetic time series data (sine wave)
def generate_sine_wave_data(n_samples=1000, seq_length=60):
    # Time steps
    time = np.linspace(0, 30, n_samples)
    
    # Generate sine wave with some noise
    data = np.sin(time) + 0.1 * np.random.randn(n_samples)
    
    # Create sequences for training
    X, y = [], []
    for i in range(len(data) - seq_length):
        X.append(data[i:i+seq_length])
        y.append(data[i+seq_length])
        
    # Convert to tensors
    X = torch.FloatTensor(np.array(X)).unsqueeze(-1)  # [batch, seq_len, feature]
    y = torch.FloatTensor(np.array(y)).unsqueeze(-1)  # [batch, feature]
    
    # Split into training and test sets
    train_size = int(0.8 * X.shape[0])
    X_train, X_test = X[:train_size], X[train_size:]
    y_train, y_test = y[:train_size], y[train_size:]
    
    return X_train, y_train, X_test, y_test, data

# Generate data
X_train, y_train, X_test, y_test, time_series_data = generate_sine_wave_data()

# Plot the data
plt.figure(figsize=(12, 4))
plt.plot(time_series_data[:200])
plt.title("Synthetic Time Series Data (Sine Wave)")
plt.xlabel("Time Steps")
plt.ylabel("Value")
plt.grid(True)
plt.show()

print(f"Training data shape: {X_train.shape}")
print(f"Training target shape: {y_train.shape}")
print(f"Testing data shape: {X_test.shape}")
print(f"Testing target shape: {y_test.shape}")

# Initialize model
time_series_transformer = TimeSeriesTransformer().to(device)

# Define loss function and optimizer
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(time_series_transformer.parameters(), lr=0.001)

# Training loop
def train(model, X_train, y_train, epochs=100, batch_size=64):
    model.train()
    train_losses = []
    
    # Create data loader
    train_data = torch.utils.data.TensorDataset(X_train, y_train)
    train_loader = torch.utils.data.DataLoader(
        train_data, batch_size=batch_size, shuffle=True
    )
    
    for epoch in range(epochs):
        epoch_loss = 0
        for batch_X, batch_y in train_loader:
            # Move data to device
            batch_X = batch_X.to(device)
            batch_y = batch_y.to(device)
            
            # Zero gradients
            optimizer.zero_grad()
            
            # Transpose for transformer: [batch, seq_len, features] -> [seq_len, batch, features]
            batch_X = batch_X.transpose(0, 1)
            
            # Forward pass
            pred = model(batch_X)
            
            # Compute loss
            loss = criterion(pred, batch_y)
            
            # Backward pass and optimization
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item() * batch_X.size(1)
        
        # Calculate average loss
        avg_loss = epoch_loss / len(train_loader.dataset)
        train_losses.append(avg_loss)
        
        if (epoch+1) % 10 == 0:
            print(f"Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.6f}")
    
    return train_losses

# Train the transformer model
train_losses = train(time_series_transformer, X_train, y_train, epochs=100)

# Plot training loss
plt.figure(figsize=(10, 5))
plt.plot(train_losses)
plt.title('Training Loss (Time Series Transformer)')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.grid(True)
plt.show()

# Evaluate on test data
def evaluate(model, X_test, y_test):
    model.eval()
    with torch.no_grad():
        # Move data to device and transpose
        X_test = X_test.to(device).transpose(0, 1)
        y_test = y_test.to(device)
        
        # Get predictions
        y_pred = model(X_test)
        
        # Compute loss
        test_loss = criterion(y_pred, y_test).item()
        
    print(f"Test Loss: {test_loss:.6f}")
    
    # Convert to numpy for plotting
    y_pred = y_pred.cpu().numpy()
    y_test = y_test.cpu().numpy()
    
    return y_pred, y_test

y_pred, y_test_np = evaluate(time_series_transformer, X_test, y_test)

# Plot predictions vs actual
plt.figure(figsize=(12, 6))
plt.plot(y_test_np[:100], label='Actual')
plt.plot(y_pred[:100], label='Predicted')
plt.title('Time Series Transformer: Predictions vs Actual')
plt.xlabel('Time Steps')
plt.ylabel('Value')
plt.legend()
plt.grid(True)
plt.show()



## 10. Why Transformers Are Taking Over: Advantages and Limitations

### Advantages of Transformers:

1. **Parallelization**: Process all elements of a sequence in parallel, unlike RNNs
2. **Capture Long-Range Dependencies**: Self-attention allows direct connections between any positions
3. **Scalability**: Transformer-based models can scale to billions of parameters
4. **Transfer Learning**: Pre-trained on large datasets and fine-tuned for specific tasks
5. **Versatility**: Applicable to many domains beyond NLP

### Limitations:

1. **Quadratic Complexity**: Self-attention has O(n²) complexity with sequence length
2. **Position Encoding Limitations**: Fixed-length positional encodings can limit handling very long sequences
3. **Training Data Requirements**: Need large amounts of data
4. **Compute Resources**: Training large transformers requires significant computational resources



In [ ]:
def compare_complexity():
    # Sequence lengths
    seq_lengths = np.arange(10, 1001, 10)
    
    # Complexity calculations
    rnn_complexity = seq_lengths  # O(n)
    transformer_complexity = seq_lengths**2  # O(n²)
    
    # Create the plot
    plt.figure(figsize=(10, 6))
    plt.plot(seq_lengths, rnn_complexity, label='RNN - O(n)')
    plt.plot(seq_lengths, transformer_complexity, label='Transformer - O(n²)')
    plt.title('Computational Complexity: RNN vs Transformer')
    plt.xlabel('Sequence Length (n)')
    plt.ylabel('Complexity (operations)')
    plt.legend()
    plt.grid(True)
    
    # Use log scale to better visualize the difference
    plt.yscale('log')
    plt.show()
    
    # Also show a comparison table
    complexity_data = {
        'Model Type': ['RNN/LSTM', 'Transformer', 'Linformer', 'Reformer', 'Performer'],
        'Time Complexity': ['O(n)', 'O(n²)', 'O(n)', 'O(n log n)', 'O(n)'],
        'Memory Usage': ['High', 'Very High', 'Moderate', 'Low', 'Low'],
        'Parallelization': ['Poor', 'Excellent', 'Excellent', 'Good', 'Excellent'],
        'Long Dependencies': ['Poor', 'Excellent', 'Good', 'Good', 'Good']
    }
    
    complexity_df = pd.DataFrame(complexity_data)
    print("Comparison of Model Architectures:")
    print(complexity_df)

compare_complexity()



## 11. Recent Developments: Efficient Transformers

Several approaches have been developed to address the quadratic complexity issue of transformers:

1. **Sparse Transformers**: Use sparse attention patterns
2. **Linformer**: Low-rank approximation of self-attention 
3. **Reformer**: Locality-sensitive hashing for efficient attention
4. **Performer**: Fast attention via orthogonal random features
5. **Longformer**: Combination of local and global attention
6. **FlashAttention**: Optimized implementation of attention computation



In [ ]:
# Visualize attention patterns of different efficient transformer variants
def visualize_attention_patterns():
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    
    # Helper function to draw attention patterns
    def draw_attn_pattern(ax, pattern_func, title):
        seq_len = 20
        matrix = np.zeros((seq_len, seq_len))
        pattern_func(matrix)
        ax.imshow(matrix, cmap='Blues')
        ax.set_title(title)
        ax.set_xlabel("Key Position")
        ax.set_ylabel("Query Position")
    
    # Pattern functions
    def full_attention(matrix):
        matrix[:] = 1
        
    def local_attention(matrix, window=3):
        for i in range(len(matrix)):
            start = max(0, i-window)
            end = min(len(matrix), i+window+1)
            matrix[i, start:end] = 1
            
    def strided_attention(matrix, stride=3):
        for i in range(len(matrix)):
            for j in range(0, len(matrix), stride):
                matrix[i, j] = 1
            matrix[i, i] = 1  # Diagonal
    
    def global_attention(matrix, global_tokens=5):
        # All tokens attend to first global_tokens
        matrix[:, :global_tokens] = 1
        # Global tokens attend to all
        matrix[:global_tokens, :] = 1
        
    def longformer_attention(matrix, window=3, global_tokens=3):
        # Local attention
        local_attention(matrix, window)
        # Global attention
        matrix[:, :global_tokens] = 1
        matrix[:global_tokens, :] = 1
        
    def random_attention(matrix, sparsity=0.8):
        rand_matrix = np.random.rand(*matrix.shape)
        matrix[:] = (rand_matrix > sparsity).astype(float)
        # Ensure diagonal is always attended to
        for i in range(len(matrix)):
            matrix[i, i] = 1
    
    # Draw patterns
    draw_attn_pattern(axes[0, 0], full_attention, "Full Attention\n(Quadratic Complexity)")
    draw_attn_pattern(axes[0, 1], local_attention, "Local Attention\n(Linear Complexity)")
    draw_attn_pattern(axes[0, 2], strided_attention, "Strided Attention")
    draw_attn_pattern(axes[1, 0], global_attention, "Global Attention")
    draw_attn_pattern(axes[1, 1], longformer_attention, "Longformer Attention")
    draw_attn_pattern(axes[1, 2], random_attention, "Sparse Random Attention")
    
    plt.tight_layout()
    plt.show()

visualize_attention_patterns()



## 12. Transformers vs. RNNs/LSTMs: A Detailed Comparison

Let's compare transformers to traditional recurrent architectures:



In [ ]:
# Create comparison table
def create_comparison_table():
    comparison_data = {
        'Feature': [
            'Parallel Computation', 
            'Handling Long Sequences', 
            'Training Efficiency',
            'Inference Speed',
            'Memory Usage',
            'Capturing Long Dependencies',
            'Training Stability',
            'Parameter Efficiency',
            'Pre-training Benefits',
            'Interpretability'
        ],
        'RNN/LSTM': [
            'Sequential only',
            'Struggles with long sequences',
            'Slower due to sequential nature',
            'Can be efficient for short sequences',
            'Memory efficient',
            'Struggles with very long dependencies',
            'Prone to vanishing/exploding gradients',
            'Often more parameter-efficient',
            'Limited benefit from pre-training',
            'Hidden states hard to interpret'
        ],
        'Transformer': [
            'Highly parallel',
            'Length limited by attention complexity',
            'Faster with parallel GPUs',
            'Constant-time inference (non-autoregressive)',
            'High memory requirements',
            'Excellent at capturing long dependencies',
            'More stable gradient flow',
            'Often needs more parameters',
            'Significant gains from pre-training',
            'Attention weights offer better interpretability'
        ]
    }
    
    comp_df = pd.DataFrame(comparison_data)
    print("Comparison between RNNs/LSTMs and Transformers:")
    display(comp_df)
    
    # Also create a visual comparison
    plt.figure(figsize=(12, 8))
    aspects = ['Parallel Processing', 'Long Dependencies', 'Training Speed', 
              'Parameter Efficiency', 'Inference Speed', 'Memory Usage']
    
    # Values from 0 to 5 for each aspect
    rnn_scores = [1, 2, 1, 4, 3, 4]
    transformer_scores = [5, 5, 4, 2, 4, 1]
    
    # Create radar chart
    angles = np.linspace(0, 2*np.pi, len(aspects), endpoint=False).tolist()
    angles += angles[:1]  # Close the loop
    
    rnn_scores += rnn_scores[:1]
    transformer_scores += transformer_scores[:1]
    aspects += aspects[:1]
    
    ax = plt.subplot(111, polar=True)
    ax.plot(angles, rnn_scores, 'b-', linewidth=2, label='RNN/LSTM')
    ax.fill(angles, rnn_scores, 'b', alpha=0.1)
    ax.plot(angles, transformer_scores, 'r-', linewidth=2, label='Transformer')
    ax.fill(angles, transformer_scores, 'r', alpha=0.1)
    
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(aspects[:-1])
    ax.set_yticks([1, 2, 3, 4, 5])
    ax.set_yticklabels(['1', '2', '3', '4', '5'])
    ax.set_ylim(0, 5)
    
    plt.legend(loc='upper right')
    plt.title('RNN/LSTM vs Transformer: Capability Comparison')
    plt.tight_layout()
    plt.show()

create_comparison_table()



## 13. Conclusion and Future Directions

Transformers have revolutionized machine learning with their ability to process sequences in parallel while capturing complex relationships. Their success stems from:

1. **Self-attention mechanism** that creates direct connections between all positions in a sequence
2. **Parallelization** that enables efficient training on modern hardware
3. **Scalability** that allows models to grow to billions of parameters
4. **Transfer learning capabilities** that enable pre-training on large datasets

### Future directions:

1. **Efficient Transformers**: Addressing the quadratic complexity issue
2. **Domain-Specific Architectures**: Adapting transformers for specialized tasks
3. **Multimodal Processing**: Connecting multiple data modalities (text, images, audio)
4. **Ethical Considerations**: Addressing biases in large pre-trained models
5. **Interpretability**: Making transformer decisions more explainable
6. **Smaller, More Efficient Models**: Making transformers accessible on resource-constrained devices

The transformer architecture has unlocked significant advances in AI capability and continues to be the foundation for cutting-edge research and applications across many domains.

Similar code found with 2 license types